
# P7 Sensitivity Suite

This notebook reruns the frozen country-projection + Louvain pipeline under the approved sensitivity specifications:

- `alt_low_end` binning
- `alt_giga` binning
- `total capacity >= 500 MW` country-filter sensitivity

The goal is to confirm broad camp-logic stability rather than exact membership invariance.


In [ ]:

from pathlib import Path
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'data' / 'Global-Solar-Power-Tracker-February-2026.xlsx').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'Global-Solar-Power-Tracker-February-2026.xlsx'
TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P7__nogit'
DATASET_RELEASE = 'Global-Solar-Power-Tracker-February-2026.xlsx'
SHEET_NAME = 'Utility-Scale (1 MW+)'
STATUS_ORDER = ['operating', 'construction', 'pre-construction', 'announced']
MAIN_SCHEME = {
    'name': 'baseline',
    'label': 'Main binning',
    'bins': [1, 5, 50, 500, np.inf],
    'labels': ['1-5', '5-50', '50-500', '500+'],
}
ALT_SCHEMES = {
    'alt_low_end': {
        'label': 'Alternative low-end split',
        'bins': [1, 5, 20, 100, np.inf],
        'labels': ['1-5', '5-20', '20-100', '100+'],
        'suffix': '__sens_altlow',
    },
    'alt_giga': {
        'label': 'Alternative giga split',
        'bins': [1, 10, 100, 1000, np.inf],
        'labels': ['1-10', '10-100', '100-1000', '1000+'],
        'suffix': '__sens_altgiga',
    },
}
COUNTRY_FILTERS = {
    'records_ge_30': {
        'label': 'Record count >= 30',
        'suffix': '__main_capacity',
    },
    'capacity_ge_500mw': {
        'label': 'Total capacity >= 500 MW',
        'suffix': '__sens_country500mw',
    },
}
WEIGHT_KINDS = {
    'count': '__secondary_count',
    'capacity': '__main_capacity',
}
SEED = 42
RESOLUTION = 1.0
TOP_K = 6
SIMILARITY = 'cosine'
EXPECTED_COUNTRIES = 94
EXPECTED_ROWS = 98942
EXPECTED_CAPACITY = 3573038.8
EXPECTED_COUNTRY500 = {
    'countries': 124,
    'rows': 99338,
    'capacity_mw': 3658030.6,
}

manifest = {
    'notebook': 'P7_Sensitivity_Suite.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'dataset_release': DATASET_RELEASE,
    'sheet_name': SHEET_NAME,
    'projection_rule': {
        'similarity': SIMILARITY,
        'top_k': TOP_K,
        'community_method': 'networkx.community.louvain_communities',
        'seed': SEED,
        'resolution': RESOLUTION,
    },
    'approved_alt_schemes': ALT_SCHEMES,
    'country_filters': COUNTRY_FILTERS,
}
(RUNTIME / 'p7_sensitivity_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Sensitivity scope:** approved alternative binning sets plus the optional `capacity >= 500 MW` country-filter sensitivity'))



## Load frozen baseline artifacts and raw source table


In [ ]:

baseline_retained = pd.read_csv(TABLES / 'p1_retained_baseline.csv', low_memory=False)
baseline_count_row = pd.read_csv(TABLES / 'p2_country_mode_count_row_normalized.csv').set_index('Country/Area')
baseline_capacity_row = pd.read_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv').set_index('Country/Area')
p3_count_membership = pd.read_csv(TABLES / 'p3_count_community_membership.csv')
p3_capacity_membership = pd.read_csv(TABLES / 'p3_capacity_community_membership.csv')
p3_network_summary = pd.read_csv(TABLES / 'p3_country_projection_network_summary.csv')

raw_util = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, engine='openpyxl')
util = raw_util.copy()
util['Country/Area'] = util['Country/Area'].astype('string').str.strip()
util['Status'] = util['Status'].astype('string').str.strip().str.lower()
util['Capacity (MW)'] = pd.to_numeric(util['Capacity (MW)'], errors='coerce')
util['Start year'] = pd.to_numeric(util['Start year'], errors='coerce')
util['Retired year'] = pd.to_numeric(util['Retired year'], errors='coerce')
util['Date Last Researched'] = pd.to_datetime(util['Date Last Researched'], errors='coerce')
util['Location accuracy'] = util['Location accuracy'].astype('string').str.strip().str.lower()
util['Phase Name'] = util['Phase Name'].fillna('--').astype(str).str.strip()
util['Country/Area'] = util['Country/Area'].fillna('Unknown').astype(str)

main_status_mask = util['Status'].isin(STATUS_ORDER)
capacity_mask = util['Capacity (MW)'].notna() & util['Capacity (MW)'].ge(1)
clean_util = util.loc[main_status_mask & capacity_mask].copy()

if baseline_retained['Country/Area'].nunique() != EXPECTED_COUNTRIES:
    raise ValueError('Frozen P1 retained scope no longer contains 94 countries.')
if len(baseline_retained) != EXPECTED_ROWS:
    raise ValueError('Frozen P1 retained scope no longer contains 98,942 rows.')
if not math.isclose(float(baseline_retained['Capacity (MW)'].sum()), EXPECTED_CAPACITY, rel_tol=0, abs_tol=1e-6):
    raise ValueError('Frozen P1 retained capacity drifted from 3,573,038.8 MW.')

display(pd.DataFrame([
    {
        'baseline_countries': int(baseline_retained['Country/Area'].nunique()),
        'baseline_rows': int(len(baseline_retained)),
        'baseline_capacity_mw': round(float(baseline_retained['Capacity (MW)'].sum()), 1),
        'clean_four_status_rows_before_country_filter': int(len(clean_util)),
        'clean_four_status_countries_before_country_filter': int(clean_util['Country/Area'].nunique()),
    }
]))



## Helper functions


In [ ]:

def split_mode(mode):
    return mode.split(' | ', 1)


def make_scenario_run_id(suffix):
    return f'{PIPELINE_VERSION}__20260320__P7{suffix}__nogit'


def row_normalize(matrix):
    row_sums = matrix.sum(axis=1).replace(0, np.nan)
    return matrix.div(row_sums, axis=0).fillna(0)


def top_share(series, n=5):
    ordered = pd.Series(series, dtype=float).sort_values(ascending=False)
    total = float(ordered.sum())
    if total <= 0:
        return np.nan
    return float(ordered.head(n).sum() / total)


def apply_country_filter(df, filter_name):
    if filter_name == 'records_ge_30':
        keep = df.groupby('Country/Area').size()
        selected = keep[keep >= 30].index
    elif filter_name == 'capacity_ge_500mw':
        keep = df.groupby('Country/Area')['Capacity (MW)'].sum()
        selected = keep[keep >= 500].index
    else:
        raise ValueError(f'Unknown country filter: {filter_name}')
    return df.loc[df['Country/Area'].isin(selected)].copy()


def assign_modes(df, bins, labels):
    out = df.copy()
    out['capacity_bin'] = pd.cut(
        out['Capacity (MW)'],
        bins=bins,
        labels=labels,
        right=False,
        include_lowest=True,
    )
    out = out.loc[out['capacity_bin'].notna()].copy()
    mode_order = [f'{status} | {label}' for status in STATUS_ORDER for label in labels]
    out['mode'] = pd.Categorical(
        out['Status'].astype(str) + ' | ' + out['capacity_bin'].astype(str),
        categories=mode_order,
        ordered=True,
    )
    tier_lookup = {label: idx + 1 for idx, label in enumerate(labels)}
    out['capacity_tier_rank'] = out['capacity_bin'].astype(str).map(tier_lookup)
    return out, mode_order, tier_lookup


def build_mode_matrix(df, mode_order, weight_kind):
    if weight_kind == 'count':
        work = df.assign(weight=1.0)
    elif weight_kind == 'capacity':
        work = df.assign(weight=df['Capacity (MW)'].astype(float))
    else:
        raise ValueError(f'Unknown weight kind: {weight_kind}')
    matrix = work.pivot_table(
        index='Country/Area',
        columns='mode',
        values='weight',
        aggfunc='sum',
        fill_value=0,
    )
    matrix = matrix.reindex(columns=mode_order, fill_value=0)
    matrix = matrix.reindex(sorted(matrix.index), axis=0)
    return matrix.astype(float)


def build_projection_graph(matrix, similarity='cosine', top_k=6):
    names = list(matrix.index)
    values = matrix.to_numpy(dtype=float)
    graph = nx.Graph()
    graph.add_nodes_from(names)

    if len(names) < 2:
        return graph

    if similarity == 'cosine':
        sim = cosine_similarity(values)
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unknown similarity: {similarity}')

    np.fill_diagonal(sim, 0)
    k = min(top_k, len(names) - 1)

    for i, source in enumerate(names):
        neighbor_idx = np.argsort(sim[i])[-k:]
        for j in neighbor_idx:
            if i == j:
                continue
            weight = float(sim[i, j])
            if weight <= 0:
                continue
            target = names[j]
            if graph.has_edge(source, target):
                graph[source][target]['weight'] = max(graph[source][target]['weight'], weight)
            else:
                graph.add_edge(source, target, weight=weight)
    return graph


def detect_communities(graph, seed=42, resolution=1.0):
    if graph.number_of_nodes() == 0:
        return pd.Series(dtype=int), [], np.nan, pd.Series(dtype=float), pd.Series(dtype=float)

    if graph.number_of_edges() == 0:
        assignment = pd.Series({node: idx + 1 for idx, node in enumerate(sorted(graph.nodes()))}, name='community')
        communities = [{node} for node in assignment.index]
        strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
        degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
        return assignment, communities, 0.0, strengths, degrees

    communities = list(nx.community.louvain_communities(graph, weight='weight', seed=seed, resolution=resolution))
    communities = sorted(communities, key=len, reverse=True)
    assignment = pd.Series(
        {node: cid for cid, members in enumerate(communities, start=1) for node in members},
        name='community',
    ).sort_index()
    strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
    degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
    modularity = float(nx.community.modularity(graph, communities, weight='weight'))
    return assignment, communities, modularity, strengths, degrees


def draft_name_from_profile(profile):
    ranked = pd.Series(profile, dtype=float).sort_values(ascending=False)
    first_mode = ranked.index[0]
    second_mode = ranked.index[1]
    first_status, first_tier = split_mode(first_mode)
    second_status, _ = split_mode(second_mode)
    first_share = float(ranked.iloc[0])
    second_share = float(ranked.iloc[1])

    if second_status != first_status and second_share >= 0.15:
        return f'Draft: {first_status} / {second_status} mixed ({first_tier}-led)'
    if first_share >= 0.45:
        return f'Draft: {first_status} {first_tier} heavy'
    if first_share >= 0.30:
        return f'Draft: {first_status} {first_tier} led'
    return f'Draft: diversified {first_status} {first_tier}'


def summarize_community_profiles(weight_kind, scenario_name, scenario_label, scenario_run_id, raw_matrix, row_matrix, assignment, strengths):
    rows = []
    tier_lookup = {}
    for mode in row_matrix.columns:
        tier = split_mode(mode)[1]
        if tier not in tier_lookup:
            tier_lookup[tier] = len(tier_lookup) + 1

    for cid in sorted(assignment.unique()):
        members = assignment[assignment == cid].index.tolist()
        profile = row_matrix.loc[members].mean(axis=0).astype(float)
        ranked = profile.sort_values(ascending=False)
        top3 = ranked.head(3)
        rows.append({
            'scenario_name': scenario_name,
            'scenario_label': scenario_label,
            'scenario_run_id': scenario_run_id,
            'weight_kind': weight_kind,
            'community': int(cid),
            'community_name_draft': draft_name_from_profile(profile),
            'countries': int(len(members)),
            'country_share': float(len(members) / len(row_matrix)),
            'raw_total_weight': float(raw_matrix.loc[members].to_numpy().sum()),
            'raw_weight_share': float(raw_matrix.loc[members].to_numpy().sum() / raw_matrix.to_numpy().sum()),
            'graph_strength_sum': float(strengths.loc[members].sum()),
            'graph_strength_share': float(strengths.loc[members].sum() / strengths.sum()),
            'dominant_mode': top3.index[0],
            'dominant_mode_share': float(top3.iloc[0]),
            'second_mode': ranked.index[1],
            'second_mode_share': float(ranked.iloc[1]),
            'top3_modes': ' / '.join([f'{mode} ({share:.2f})' for mode, share in top3.items()]),
            'top3_share': float(top3.sum()),
            'dominant_status': split_mode(top3.index[0])[0],
            'dominant_tier': split_mode(top3.index[0])[1],
            'dominant_tier_rank': int(tier_lookup[split_mode(top3.index[0])[1]]),
            'interpretable': bool(top3.iloc[0] >= 0.25 and top3.sum() >= 0.60),
            'members_sample': ', '.join(members[:8]),
        })
        for mode in row_matrix.columns:
            rows[-1][f'profile__{mode}'] = float(profile[mode])
    return pd.DataFrame(rows).sort_values(['countries', 'raw_total_weight'], ascending=[False, False]).reset_index(drop=True)


def dominant_country_signatures(weight_kind, scenario_name, scenario_label, scenario_run_id, raw_matrix, row_matrix):
    tier_lookup = {}
    for mode in row_matrix.columns:
        tier = split_mode(mode)[1]
        if tier not in tier_lookup:
            tier_lookup[tier] = len(tier_lookup) + 1
    dominant_mode = row_matrix.idxmax(axis=1)
    dominant_mode_share = row_matrix.max(axis=1)
    out = pd.DataFrame({
        'Country/Area': row_matrix.index,
        'scenario_name': scenario_name,
        'scenario_label': scenario_label,
        'scenario_run_id': scenario_run_id,
        'weight_kind': weight_kind,
        'dominant_mode': dominant_mode.values,
        'dominant_mode_share': dominant_mode_share.values,
        'raw_total_weight_country': raw_matrix.sum(axis=1).reindex(row_matrix.index).astype(float).values,
    })
    out['dominant_status'] = out['dominant_mode'].map(lambda x: split_mode(x)[0])
    out['dominant_tier'] = out['dominant_mode'].map(lambda x: split_mode(x)[1])
    out['dominant_tier_rank'] = out['dominant_tier'].map(tier_lookup)
    return out.sort_values('raw_total_weight_country', ascending=False).reset_index(drop=True)


def community_signature_overlap(left_profiles, right_profiles, top_k=5):
    left_top = left_profiles.sort_values(['countries', 'dominant_mode_share'], ascending=[False, False]).head(top_k)
    right_top = right_profiles.sort_values(['countries', 'dominant_mode_share'], ascending=[False, False]).head(top_k)
    left_signatures = set(zip(left_top['dominant_status'], left_top['dominant_tier_rank']))
    right_signatures = set(zip(right_top['dominant_status'], right_top['dominant_tier_rank']))
    if not left_signatures or not right_signatures:
        return np.nan
    return float(len(left_signatures & right_signatures) / min(len(left_signatures), len(right_signatures)))


def dominant_signature_string(profile_df, field):
    ordered = profile_df.sort_values(['countries', 'dominant_mode_share'], ascending=[False, False])
    return '; '.join(str(value) for value in ordered[field].tolist())


def run_country_projection(df, scheme_name, scheme_label, bins, labels, weight_kind, scenario_run_id):
    assigned, mode_order, tier_lookup = assign_modes(df, bins=bins, labels=labels)
    raw_matrix = build_mode_matrix(assigned, mode_order, weight_kind=weight_kind)
    row_matrix = row_normalize(raw_matrix)
    graph = build_projection_graph(row_matrix, similarity=SIMILARITY, top_k=TOP_K)
    assignment, communities, modularity, strengths, degrees = detect_communities(graph, seed=SEED, resolution=RESOLUTION)
    profiles = summarize_community_profiles(
        weight_kind=weight_kind,
        scenario_name=scheme_name,
        scenario_label=scheme_label,
        scenario_run_id=scenario_run_id,
        raw_matrix=raw_matrix,
        row_matrix=row_matrix,
        assignment=assignment,
        strengths=strengths,
    )
    country_signatures = dominant_country_signatures(
        weight_kind=weight_kind,
        scenario_name=scheme_name,
        scenario_label=scheme_label,
        scenario_run_id=scenario_run_id,
        raw_matrix=raw_matrix,
        row_matrix=row_matrix,
    )
    return {
        'assigned_df': assigned,
        'mode_order': mode_order,
        'tier_lookup': tier_lookup,
        'raw_matrix': raw_matrix,
        'row_matrix': row_matrix,
        'graph': graph,
        'assignment': assignment,
        'communities': communities,
        'modularity': modularity,
        'strengths': strengths,
        'degrees': degrees,
        'profiles': profiles,
        'country_signatures': country_signatures,
        'countries': int(len(row_matrix)),
        'rows': int(len(assigned)),
        'capacity_mw': float(assigned['Capacity (MW)'].sum()),
        'top5_strength_share': float(top_share(strengths, 5)),
        'interpretable_communities': int(profiles['interpretable'].sum()),
        'largest_community_share': float(max((len(c) for c in communities), default=0) / max(len(assignment), 1)),
    }



## Reproduce the frozen baseline with the same projection and community rules


In [ ]:

expected_p3 = p3_network_summary.set_index('weight_kind')[['communities', 'modularity']].to_dict('index')
loaded_p3_assignments = {
    'count': p3_count_membership.set_index('Country/Area')['community'].sort_index(),
    'capacity': p3_capacity_membership.set_index('Country/Area')['community'].sort_index(),
}

baseline_objects = {}
for weight_kind in ['count', 'capacity']:
    scenario_run_id = make_scenario_run_id(WEIGHT_KINDS[weight_kind])
    baseline_objects[weight_kind] = run_country_projection(
        baseline_retained.copy(),
        scheme_name='baseline',
        scheme_label='Main binning',
        bins=MAIN_SCHEME['bins'],
        labels=MAIN_SCHEME['labels'],
        weight_kind=weight_kind,
        scenario_run_id=scenario_run_id,
    )
    observed = baseline_objects[weight_kind]
    expected = expected_p3[weight_kind]
    if int(len(observed['communities'])) != int(expected['communities']):
        raise ValueError(f'{weight_kind}: baseline rerun community count drifted from P3.')
    if not math.isclose(float(observed['modularity']), float(expected['modularity']), rel_tol=0, abs_tol=1e-12):
        raise ValueError(f'{weight_kind}: baseline rerun modularity drifted from P3.')
    common = observed['assignment'].index.intersection(loaded_p3_assignments[weight_kind].index)
    if adjusted_rand_score(observed['assignment'].loc[common], loaded_p3_assignments[weight_kind].loc[common]) != 1.0:
        raise ValueError(f'{weight_kind}: baseline rerun no longer matches frozen P3 community assignments.')

baseline_reference = []
for weight_kind, obj in baseline_objects.items():
    baseline_reference.append({
        'weight_kind': weight_kind,
        'countries': obj['countries'],
        'rows': obj['rows'],
        'capacity_mw': round(obj['capacity_mw'], 1),
        'communities': len(obj['communities']),
        'modularity': obj['modularity'],
        'top5_strength_share': obj['top5_strength_share'],
        'interpretable_communities': obj['interpretable_communities'],
    })

display(pd.DataFrame(baseline_reference))



## Run approved alternative binning schemes and compute stability summaries


In [ ]:

binning_summary_rows = []
scenario_profile_outputs = {}

for scheme_name, scheme_meta in ALT_SCHEMES.items():
    scenario_profiles = []
    for weight_kind in ['count', 'capacity']:
        scenario_run_id = make_scenario_run_id(scheme_meta['suffix'])
        result = run_country_projection(
            baseline_retained.copy(),
            scheme_name=scheme_name,
            scheme_label=scheme_meta['label'],
            bins=scheme_meta['bins'],
            labels=scheme_meta['labels'],
            weight_kind=weight_kind,
            scenario_run_id=scenario_run_id,
        )
        baseline = baseline_objects[weight_kind]
        common = baseline['assignment'].index.intersection(result['assignment'].index)
        ari = adjusted_rand_score(baseline['assignment'].loc[common], result['assignment'].loc[common]) if len(common) > 1 else np.nan

        baseline_top20 = baseline['country_signatures']['Country/Area'].head(20)
        merged_country = baseline['country_signatures'][['Country/Area', 'dominant_status', 'dominant_tier_rank']].merge(
            result['country_signatures'][['Country/Area', 'dominant_status', 'dominant_tier_rank']],
            on='Country/Area',
            suffixes=('_baseline', '_alt'),
        )
        merged_top20 = merged_country.loc[merged_country['Country/Area'].isin(baseline_top20)].copy()
        status_stability_top20 = float((merged_top20['dominant_status_baseline'] == merged_top20['dominant_status_alt']).mean()) if len(merged_top20) else np.nan
        tier_stability_top20 = float((merged_top20['dominant_tier_rank_baseline'] == merged_top20['dominant_tier_rank_alt']).mean()) if len(merged_top20) else np.nan
        dominant_profile_stability_top20 = float(((merged_top20['dominant_status_baseline'] == merged_top20['dominant_status_alt']) & (merged_top20['dominant_tier_rank_baseline'] == merged_top20['dominant_tier_rank_alt'])).mean()) if len(merged_top20) else np.nan
        camp_signature_overlap_top5 = community_signature_overlap(baseline['profiles'], result['profiles'], top_k=5)

        binning_summary_rows.append({
            'scenario_name': scheme_name,
            'scenario_label': scheme_meta['label'],
            'scenario_run_id': scenario_run_id,
            'weight_kind': weight_kind,
            'countries': result['countries'],
            'rows': result['rows'],
            'capacity_mw': round(result['capacity_mw'], 1),
            'communities': int(len(result['communities'])),
            'modularity': float(result['modularity']),
            'top5_strength_share': float(result['top5_strength_share']),
            'interpretable_communities': int(result['interpretable_communities']),
            'largest_community_share': float(result['largest_community_share']),
            'common_countries_vs_baseline': int(len(common)),
            'ari_vs_baseline': float(ari),
            'status_stability_top20': float(status_stability_top20),
            'tier_stability_top20': float(tier_stability_top20),
            'dominant_profile_stability_top20': float(dominant_profile_stability_top20),
            'camp_signature_overlap_top5': float(camp_signature_overlap_top5),
            'dominant_status_signature': dominant_signature_string(result['profiles'], 'dominant_status'),
            'dominant_tier_signature': dominant_signature_string(result['profiles'], 'dominant_tier'),
        })
        scenario_profiles.append(result['profiles'])
    scenario_profile_outputs[scheme_name] = pd.concat(scenario_profiles, ignore_index=True)

binning_summary = pd.DataFrame(binning_summary_rows).sort_values(['scenario_name', 'weight_kind']).reset_index(drop=True)
scenario_profile_outputs['alt_low_end'].to_csv(TABLES / 'p7_altlow_community_outputs.csv', index=False)
scenario_profile_outputs['alt_giga'].to_csv(TABLES / 'p7_altgiga_community_outputs.csv', index=False)
binning_summary.to_csv(TABLES / 'p7_binning_stability_summary.csv', index=False)

if len(binning_summary) != 4:
    raise ValueError('P7 binning summary should contain 4 rows (2 schemes x 2 weighting views).')
if int((binning_summary['countries'] == EXPECTED_COUNTRIES).sum()) != 4:
    raise ValueError('Alternative binning should not change the retained 94-country baseline scope.')
if int((binning_summary['rows'] == EXPECTED_ROWS).sum()) != 4:
    raise ValueError('Alternative binning should not change the retained row count.')
if int(np.isclose(binning_summary['capacity_mw'], EXPECTED_CAPACITY).sum()) != 4:
    raise ValueError('Alternative binning should not change retained total capacity.')
if binning_summary[['ari_vs_baseline', 'status_stability_top20', 'tier_stability_top20', 'dominant_profile_stability_top20', 'camp_signature_overlap_top5']].isna().any().any():
    raise ValueError('Binning stability summary contains unexpected NaN values in key QA fields.')

display(binning_summary)



## Run the optional country-filter sensitivity with the frozen main binning


In [ ]:

country500_df = apply_country_filter(clean_util, 'capacity_ge_500mw')
country500_assigned, _, _ = assign_modes(country500_df, bins=MAIN_SCHEME['bins'], labels=MAIN_SCHEME['labels'])

if country500_assigned['Country/Area'].nunique() != EXPECTED_COUNTRY500['countries']:
    raise ValueError('capacity >= 500 MW sensitivity countries drifted from the precheck baseline.')
if len(country500_assigned) != EXPECTED_COUNTRY500['rows']:
    raise ValueError('capacity >= 500 MW sensitivity row count drifted from the precheck baseline.')
if not math.isclose(float(country500_assigned['Capacity (MW)'].sum()), EXPECTED_COUNTRY500['capacity_mw'], rel_tol=0, abs_tol=1e-6):
    raise ValueError('capacity >= 500 MW sensitivity retained capacity drifted from the precheck baseline.')

countryfilter_rows = []
for weight_kind in ['count', 'capacity']:
    scenario_run_id = make_scenario_run_id(COUNTRY_FILTERS['capacity_ge_500mw']['suffix'])
    result = run_country_projection(
        country500_df.copy(),
        scheme_name='capacity_ge_500mw',
        scheme_label=COUNTRY_FILTERS['capacity_ge_500mw']['label'],
        bins=MAIN_SCHEME['bins'],
        labels=MAIN_SCHEME['labels'],
        weight_kind=weight_kind,
        scenario_run_id=scenario_run_id,
    )
    baseline = baseline_objects[weight_kind]
    common = baseline['assignment'].index.intersection(result['assignment'].index)
    baseline_top20 = baseline['country_signatures']['Country/Area'].head(20)
    merged_country = baseline['country_signatures'][['Country/Area', 'dominant_mode', 'dominant_status', 'dominant_tier_rank']].merge(
        result['country_signatures'][['Country/Area', 'dominant_mode', 'dominant_status', 'dominant_tier_rank']],
        on='Country/Area',
        suffixes=('_baseline', '_alt'),
    )
    merged_top20 = merged_country.loc[merged_country['Country/Area'].isin(baseline_top20)].copy()
    countryfilter_rows.append({
        'filter_name': 'capacity_ge_500mw',
        'filter_label': COUNTRY_FILTERS['capacity_ge_500mw']['label'],
        'scenario_run_id': scenario_run_id,
        'weight_kind': weight_kind,
        'countries': int(result['countries']),
        'rows': int(result['rows']),
        'capacity_mw': round(result['capacity_mw'], 1),
        'communities': int(len(result['communities'])),
        'modularity': float(result['modularity']),
        'top5_strength_share': float(result['top5_strength_share']),
        'interpretable_communities': int(result['interpretable_communities']),
        'common_countries_vs_baseline': int(len(common)),
        'added_countries_vs_baseline': int(len(result['assignment'].index.difference(baseline['assignment'].index))),
        'removed_countries_vs_baseline': int(len(baseline['assignment'].index.difference(result['assignment'].index))),
        'ari_vs_baseline_on_common': float(adjusted_rand_score(baseline['assignment'].loc[common], result['assignment'].loc[common])) if len(common) > 1 else np.nan,
        'dominant_mode_stability_top20': float((merged_top20['dominant_mode_baseline'] == merged_top20['dominant_mode_alt']).mean()) if len(merged_top20) else np.nan,
        'status_stability_top20': float((merged_top20['dominant_status_baseline'] == merged_top20['dominant_status_alt']).mean()) if len(merged_top20) else np.nan,
        'tier_stability_top20': float((merged_top20['dominant_tier_rank_baseline'] == merged_top20['dominant_tier_rank_alt']).mean()) if len(merged_top20) else np.nan,
        'camp_signature_overlap_top5': float(community_signature_overlap(baseline['profiles'], result['profiles'], top_k=5)),
        'dominant_status_signature': dominant_signature_string(result['profiles'], 'dominant_status'),
        'dominant_tier_signature': dominant_signature_string(result['profiles'], 'dominant_tier'),
    })

countryfilter_summary = pd.DataFrame(countryfilter_rows).sort_values('weight_kind').reset_index(drop=True)
countryfilter_summary.to_csv(TABLES / 'p7_countryfilter_sensitivity_summary.csv', index=False)

if len(countryfilter_summary) != 2:
    raise ValueError('Country-filter sensitivity summary should contain 2 rows (count and capacity).')
if countryfilter_summary[['ari_vs_baseline_on_common', 'dominant_mode_stability_top20', 'status_stability_top20', 'tier_stability_top20', 'camp_signature_overlap_top5']].isna().any().any():
    raise ValueError('Country-filter sensitivity summary contains unexpected NaN values in key QA fields.')

display(countryfilter_summary)



## Write compact report and final QA summary


In [ ]:

count_binning_mean_ari = float(binning_summary.loc[binning_summary['weight_kind'] == 'count', 'ari_vs_baseline'].mean())
capacity_binning_mean_ari = float(binning_summary.loc[binning_summary['weight_kind'] == 'capacity', 'ari_vs_baseline'].mean())
count_status_min = float(binning_summary.loc[binning_summary['weight_kind'] == 'count', 'status_stability_top20'].min())
capacity_status_min = float(binning_summary.loc[binning_summary['weight_kind'] == 'capacity', 'status_stability_top20'].min())
binning_pass = bool((binning_summary['ari_vs_baseline'] >= 0.25).all() and (binning_summary['status_stability_top20'] >= 0.80).all())

report_lines = [
    '# P7 Sensitivity Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    f'- dataset_release: `{DATASET_RELEASE}`',
    '- baseline reference: frozen `P1` retained scope plus frozen `P3` community outputs',
    '',
    '## Frozen rules carried forward',
    '- projection: row-based country projection using `cosine` similarity and `top_k=6`',
    '- community detection: `networkx.community.louvain_communities` with `weight=weight`, `seed=42`, `resolution=1.0`',
    '- approved sensitivity set only: `alt_low_end`, `alt_giga`, and optional `capacity >= 500 MW` country filter',
    '',
    '## Baseline alignment check',
    f'- baseline retained scope reproduced: {EXPECTED_COUNTRIES} countries, {EXPECTED_ROWS:,} rows, {EXPECTED_CAPACITY:,.1f} MW',
    '- baseline rerun exactly matches frozen `P3` modularity and community assignments in both count and capacity views',
    '',
    '## Alternative binning results',
]
for row in binning_summary.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: {row.scenario_name} -> communities={row.communities}, modularity={row.modularity:.6f}, ARI={row.ari_vs_baseline:.6f}, top-20 dominant-status stability={row.status_stability_top20:.2f}, top-20 tier-rank stability={row.tier_stability_top20:.2f}, top-20 dominant-profile stability={row.dominant_profile_stability_top20:.2f}, camp-signature overlap={row.camp_signature_overlap_top5:.2f}'
    )
report_lines.extend([
    '',
    '## Country-filter sensitivity',
])
for row in countryfilter_summary.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: capacity >= 500 MW -> countries={row.countries}, rows={row.rows}, modularity={row.modularity:.6f}, ARI on common countries={row.ari_vs_baseline_on_common:.6f}, exact dominant-mode stability on baseline top-20={row.dominant_mode_stability_top20:.2f}, status stability={row.status_stability_top20:.2f}, tier stability={row.tier_stability_top20:.2f}, camp-signature overlap={row.camp_signature_overlap_top5:.2f}'
    )
report_lines.extend([
    '',
    '## Compact interpretation',
    f'- mean ARI baseline-vs-alt binning = {count_binning_mean_ari:.2f} (count) and {capacity_binning_mean_ari:.2f} (capacity)',
    f'- minimum top-20 dominant-status stability across approved alternative binnings = {count_status_min:.2f} (count) and {capacity_status_min:.2f} (capacity)',
    f'- broad logic stability under approved alternative binnings: {'pass' if binning_pass else 'review manually'}',
    '- exact mode labels are not directly comparable across alternative binning schemes; therefore dominant-profile stability is evaluated using dominant status plus tier rank rather than raw mode strings',
    '- the optional country-filter sensitivity is descriptive support only and does not replace the main `record count >= 30` inclusion rule',
    '',
    '## Interpretation boundary',
    '- This batch tests whether broad camp logic survives approved alternatives.',
    '- It does not require exact country-by-country membership invariance.',
    '- It does not change the main analysis scope, main weighting choice, or the frozen capacity-first interpretation hierarchy.',
    '',
    '## Output inventory',
    '- `artifacts/tables/p7_binning_stability_summary.csv`',
    '- `artifacts/tables/p7_altlow_community_outputs.csv`',
    '- `artifacts/tables/p7_altgiga_community_outputs.csv`',
    '- `artifacts/tables/p7_countryfilter_sensitivity_summary.csv`',
    '- `artifacts/reports/p7_sensitivity_report.md`',
    '- `artifacts/runtime/p7_sensitivity_manifest.json`',
])

(REPORTS / 'p7_sensitivity_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
display(Markdown((REPORTS / 'p7_sensitivity_report.md').read_text(encoding='utf-8')))
